<a href="https://colab.research.google.com/github/Lucaaa31/Anomaly-Segmentation/blob/master/notebooks/Step8_with_Temperature_v2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>


# Step 8 Anomaly Segmentation
---


# Settings


In [ ]:
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Anomaly-Segmentation
#!git pull origin master

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
/content/drive/.shortcut-targets-by-id/1Pz7ReDC4oIzyB7KLXs9SnUMvmnDqYbyB/Anomaly-Segmentation


## Dependencies

In [ ]:
!pip install --upgrade-strategy only-if-needed -r requirements.txt

Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/base_command.py", line 179, in exc_logging_wrapper
    status = run_func(*args)
             ^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/cli/req_command.py", line 67, in wrapper
    return func(self, options, args)
           ^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 447, in run
    conflicts = self._determine_conflicts(to_install)
                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/commands/install.py", line 578, in _determine_conflicts
    return check_install_conflicts(to_install)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/pip/_internal/operations/check.py", line 101, in check_install_conflicts
    package_set, _ = create_package_set_from_installed()
              

KeyboardInterrupt: 

## Imports and random seeds

In [ ]:
import os
import glob
import yaml
import random
import warnings
import importlib
from torch.amp.autocast_mode import autocast
import numpy as np
import torch
import torch.nn.functional as F
from PIL import Image
from torchvision.transforms import Compose, Resize, ToTensor
from lightning import seed_everything
from huggingface_hub import hf_hub_download
from huggingface_hub.utils import RepositoryNotFoundError
from ood_metrics import fpr_at_95_tpr
from sklearn.metrics import average_precision_score
import matplotlib.pyplot as plt
import torchvision.transforms.functional as F_tf
import torchvision.transforms as T
from torch.utils.data import TensorDataset, DataLoader


# Utils
from utils.build import build_model_and_data
from data.load_eomt.cityscapes_semantic import CityscapesSemantic
from utils.temperature_scaling import ModelWithTemperature
from utils.class_remap import remap_coco_logits_to_cs, common_target_remap


seed_everything(42, verbose=False)
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
torch.backends.cudnn.deterministic = True
torch.backends.cudnn.benchmark = True

---
# Evaluation
## Configuration


In [ ]:
dataset_to_use = "RoadAnomaly21"  # @param ["fs_static", "RoadAnomaly21", "RoadAnomaly", "FS_LostFound_full", "RoadObsticle21"]
methods = "RbA"                   # @param ["MSP", "max_logit", "max_entropy", "RbA"]

project_root  = "/content/drive/MyDrive/Anomaly-Segmentation"

data_path     = project_root + f"/dataset/Anomaly_Validation_Dataset/{dataset_to_use}"
input_pattern = data_path + "/images/*"

model_type = "cityscapes"  # @param ["coco", "cityscapes", "phased", "full"]

_CONFIGS_ROOT = project_root + "/configs/dinov2"

if model_type == "coco":
    _cfg      = _CONFIGS_ROOT + "/coco/panoptic/eomt_base_640_2x.yaml"
    _ckpt     = project_root  + "/models/eomt_eval/eomt_coco.bin"
    _override = None
elif model_type == "cityscapes":
    _cfg      = _CONFIGS_ROOT + "/cityscapes/semantic/eomt_base_640.yaml"
    _ckpt     = project_root  + "/models/eomt_eval/eomt_cityscapes.bin"
    _override = None
else:
    _cfg      = _CONFIGS_ROOT + "/cityscapes/semantic/eomt_base_640.yaml"
    _ckpt     = project_root  + f"/models/coco_finetune/{model_type}/eomt_finetuned_{model_type}.bin"
    _override = {"img_size": (640, 640)}



# temperature controls how the anomaly scores are computed during INFERENCE.
#  "none"  -> T = 1.0 (no scaling)
#  numeric -> use that fixed T
#  "best"  -> sweep T over T_GRID on THIS anomaly dataset and pick the T
#             that maximises AuPRC (FPR95 as tie-breaker).  <-- correct for Step 8
temperature = "none"  # @param ["none", "0.5", "0.75", "1.1", "best"]

# Grid of temperatures tried when temperature == "best".
T_GRID = [0.5, 0.75, 0.9, 1.0, 1.1, 1.25, 1.5, 2.0, 2.5]  # @param

device = "cuda" if torch.cuda.is_available() else "cpu"

class Args:
    def __init__(self):
        self.input   = input_pattern
        self.datadir = data_path
        self.method  = methods
        self.model = model_type
        self.temperature = temperature

args = Args()
print(f"model={model_type}  dataset={dataset_to_use}  method={methods}  temperature={temperature}")
print(f"ckpt:   {_ckpt}")
print(f"config: {_cfg}")

model=cityscapes  dataset=RoadAnomaly21  method=RbA  temperature=none
ckpt:   /content/drive/MyDrive/Anomaly-Segmentation/models/eomt_eval/eomt_cityscapes.bin
config: /content/drive/MyDrive/Anomaly-Segmentation/configs/dinov2/cityscapes/semantic/eomt_base_640.yaml


We build the model and wrap it in the ModelWithTemperature Class:

In [ ]:
print(f"Building the eomt-{model_type}")
model, meta = build_model_and_data(_cfg, _ckpt, data_path, device, setup_data=False, data_overrides=_override)
print(f"  num_classes={meta.num_classes}  img_size={meta.img_size}")
NUM_CLASSES = meta.num_classes
IMG_SIZE = meta.img_size
temperature_model = ModelWithTemperature(model).to(device)

# start from T = 1.0 (no scaling); will be overwritten below if needed
temperature_model.temperature = torch.nn.Parameter(torch.tensor([1.0]).to(device))

Building the eomt-cityscapes


model.safetensors:   0%|          | 0.00/346M [00:00<?, ?B/s]

KeyboardInterrupt: 

## Utils methods


In [ ]:
def infer_panoptic(img, target):
    with torch.no_grad(), autocast(dtype=torch.float16, device_type="cuda"):
        imgs = [img.to(device)]
        img_sizes = [img.shape[-2:] for img in imgs]

        transformed_imgs = model.resize_and_pad_imgs_instance_panoptic(imgs)
        mask_logits_per_layer, class_logits_per_layer = model(transformed_imgs)

        mask_logits = F.interpolate(
            mask_logits_per_layer[-1], model.img_size, mode="bilinear"
        )
        mask_logits = model.revert_resize_and_pad_logits_instance_panoptic(
            mask_logits, img_sizes
        )

        preds = model.to_per_pixel_preds_panoptic(
            mask_logits,
            class_logits_per_layer[-1],
            model.stuff_classes,
            model.mask_thresh,
            model.overlap_thresh,
        )[0].cpu()

    pred = preds.numpy()
    sem_pred, inst_pred = pred[..., 0], pred[..., 1]

    target_seg = model.to_per_pixel_targets_panoptic([target])[0].cpu().numpy()
    sem_target, inst_target = target_seg[..., 0], target_seg[..., 1]

    cls_logits = class_logits_per_layer[-1][0][:, :NUM_CLASSES].float()
    masks_probs = torch.sigmoid(mask_logits[0]).float()

    num_q = masks_probs.shape[0]
    H, W = masks_probs.shape[1], masks_probs.shape[2]

    semantic_logits = torch.mm(
        cls_logits.t(),
        masks_probs.view(num_q, -1),
    ).view(NUM_CLASSES, H, W)

    return sem_pred, inst_pred, sem_target, inst_target, semantic_logits, cls_logits, masks_probs

In [ ]:

def draw_black_border(sem, inst, mapping):
    h, w = sem.shape
    out = np.zeros((h, w, 3))
    for s in np.unique(sem):
        out[sem == s] = mapping[s]

    combined = sem.astype(np.int64) * 100000 + inst.astype(np.int64)
    border = np.zeros((h, w), dtype=bool)
    border[1:, :] |= combined[1:, :] != combined[:-1, :]
    border[:-1, :] |= combined[1:, :] != combined[:-1, :]
    border[:, 1:] |= combined[:, 1:] != combined[:, :-1]
    border[:, :-1] |= combined[:, 1:] != combined[:, :-1]
    out[border] = 0
    return out


def plot_panoptic_results(img, sem_pred, inst_pred, sem_target, inst_target):
    all_ids = np.union1d(np.unique(sem_pred), np.unique(sem_target))
    mapping = {
        s: (
            [0, 0, 0]
            if s == -1 or s == model.num_classes
            else plt.cm.hsv(i / len(all_ids))[:3]
        )
        for i, s in enumerate(all_ids)
    }

    vis_pred = draw_black_border(sem_pred, inst_pred, mapping)
    vis_target = draw_black_border(sem_target, inst_target, mapping)

    img_np = (
        img.cpu().numpy().transpose(1, 2, 0) if img.dim() == 3 else img.cpu().numpy()
    )

    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    axes[0].imshow(img_np)
    axes[0].set_title("Input")
    axes[1].imshow(vis_pred)
    axes[1].set_title("Prediction")
    axes[2].imshow(vis_target)
    axes[2].set_title("Target")

    for ax in axes:
        ax.axis("off")

    plt.tight_layout()
    plt.show()

## Post hoc methods

In [ ]:
def get_msp(logits):

    probs = F.softmax(logits.float(), dim=0)
    msp = probs.max(dim=0).values              # [H, W]
    return (1.0 - msp).detach().cpu().numpy().astype("float32")


def get_maxlogit(logits):
    max_logit = logits.float().max(dim=0).values
    return (-max_logit).detach().cpu().numpy().astype("float32")


def get_entropy(logits):

    logits = logits.float()
    probs = F.softmax(logits, dim=0)
    num_classes = logits.shape[0]
    entropy = -(probs * (probs + 1e-7).log()).sum(dim=0)   # [H, W]
    norm_entropy = entropy / torch.log(
        torch.tensor(num_classes, dtype=torch.float32, device=logits.device)
    )
    return norm_entropy.detach().cpu().numpy().astype("float32")


def get_Rba(cls_logits, masks_probs):


    cls_logits  = cls_logits.float()
    masks_probs = masks_probs.float()

    query_class_score = F.softmax(cls_logits, dim=1).max(dim=1).values

    rba_score = (query_class_score[:, None, None] * masks_probs).max(dim=0).values

    return (1.0 - rba_score).detach().cpu().numpy().astype("float32")

## Anomaly score for one image (given a temperature)
This helper applies a temperature `T` to the model outputs and returns the per-pixel anomaly score for the selected post-hoc method. It is used both for normal inference and for the `best`-temperature search, so the scoring logic lives in a single place.

In [ ]:
def anomaly_score_from_outputs(result_logits, cls_logits_raw, masks_probs_raw, T, method):
    """Compute the anomaly score map for one image at temperature T.

    result_logits   : semantic logits  [C, H, W]   (already remapped if COCO)
    cls_logits_raw  : query class logits [Q, C(+1)]
    masks_probs_raw : query mask probs   [Q, H, W]
    T               : scalar temperature (float or 0-dim tensor)
    method          : one of "MSP", "max_logit", "max_entropy", "RbA"
    """
    scaled_logits = result_logits / T
    match method:
        case "MSP":
            return get_msp(scaled_logits)
        case "max_logit":
            return get_maxlogit(scaled_logits)
        case "max_entropy":
            return get_entropy(scaled_logits)
        case "RbA":
            # for RbA temperature is applied to the query class logits
            return get_Rba(cls_logits_raw / T, masks_probs_raw)
    raise ValueError(f"Unknown method: {method}")

## Preprocessing

In [ ]:
def input_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.BILINEAR)
    img_tensor = F_tf.to_tensor(img_resized)
    return (img_tensor * 255).to(torch.uint8).to(device)

def target_transform(img):
    img_resized = F_tf.resize(img, IMG_SIZE, interpolation=T.InterpolationMode.NEAREST)
    return np.array(img_resized)

## Anomaly GT remap helper
Same dataset-specific OoD label remapping you already had, factored out so it can be reused by the collection pass and the temperature search.

In [ ]:
def remap_ood_gt(target_np, pathGT):
    ood_gts = target_np.copy()
    if "RoadAnomaly" in pathGT:
        ood_gts = np.where((ood_gts == 2), 1, ood_gts)
    if "FS_LostFound_full" in pathGT:
        ood_gts = np.where((ood_gts == 0), 255, ood_gts)
        ood_gts = np.where((ood_gts == 1), 0, ood_gts)
        ood_gts = np.where((ood_gts > 1) & (ood_gts < 201), 1, ood_gts)
    if "Streethazard" in pathGT:
        ood_gts = np.where((ood_gts == 14), 255, ood_gts)
        ood_gts = np.where((ood_gts < 20), 0, ood_gts)
        ood_gts = np.where((ood_gts == 255), 1, ood_gts)
    return ood_gts


def gt_path_for(path):
    pathGT = path.replace("images", "labels_masks")
    if "RoadObsticle21" in pathGT:
        pathGT = pathGT.replace("webp", "png")
    if "fs_static" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    if "RoadAnomaly" in pathGT:
        pathGT = pathGT.replace("jpg", "png")
    return pathGT

---
# Build the full temperature table in one go

This section reproduces the Step-8 temperature table automatically. For each anomaly
dataset it runs the model **once** to cache the logits (PRO TIP), then evaluates every
requested temperature on those cached logits. It works for any post-hoc method, but the
PDF table uses **MSP**, so `TABLE_METHOD` is set to `"MSP"` below.

For each (dataset, temperature) cell it reports AuPRC and FPR@95TPR. The `best` row uses,
per dataset, the temperature from `T_GRID` that maximises AuPRC on that dataset.

**Note:** mIoU is not recomputed here (it does not depend on temperature or post-hoc
method, only on the weights). Fill the mIoU column from your earlier Step-8 evaluation.


In [ ]:
def cache_dataset(ds_name):
    """Run the model once over one anomaly dataset and cache per-image outputs.
    Returns a list of dicts (same structure as `cached_outputs`)."""
    ds_data_path = project_root + f"/dataset/Anomaly_Validation_Dataset/{ds_name}"
    ds_input_pattern = ds_data_path + "/images/*"

    out = []
    for path in glob.glob(os.path.expanduser(str(ds_input_pattern))):
        raw_image = Image.open(path).convert('RGB')
        images = input_transform(raw_image)

        pathGT = gt_path_for(path)
        raw_target = Image.open(pathGT).convert('L')
        target_np = target_transform(raw_target)

        unique_ids = np.unique(target_np)
        unique_ids = unique_ids[unique_ids != 0]

        masks, labels = [], []
        for s_id in unique_ids:
            masks.append(target_np == s_id)
            labels.append(s_id)

        if len(masks) > 0:
            target_dict = {
                "masks": torch.from_numpy(np.stack(masks)).bool().to(device),
                "labels": torch.from_numpy(np.array(labels)).long().to(device),
            }
        else:
            target_dict = {
                "masks": torch.zeros((0, target_np.shape[0], target_np.shape[1]), dtype=torch.bool).to(device),
                "labels": torch.zeros((0,), dtype=torch.long).to(device),
            }

        (sem_pred, inst_pred, sem_target, inst_target,
         result_logits, cls_logits_raw, masks_probs_raw) = infer_panoptic(images, target_dict)

        if args.model == "coco":
            result_logits = remap_coco_logits_to_cs(result_logits)

        ood_gts = remap_ood_gt(target_np, pathGT)
        if 1 not in np.unique(ood_gts):
            del result_logits, cls_logits_raw, masks_probs_raw
            torch.cuda.empty_cache()
            continue

        out.append({
            "result_logits":   result_logits.detach().cpu(),
            "cls_logits_raw":  cls_logits_raw.detach().cpu(),
            "masks_probs_raw": masks_probs_raw.detach().cpu(),
            "ood_gts":         ood_gts,
        })
        del result_logits, cls_logits_raw, masks_probs_raw
        torch.cuda.empty_cache()

    return out

In [ ]:
def evaluate_cached(cache, T, method):
    """Same as evaluate_at_temperature but on an explicit cache + method."""
    anomaly_score_list, ood_gts_list = [], []
    for item in cache:
        result_logits   = item["result_logits"].to(device)
        cls_logits_raw  = item["cls_logits_raw"].to(device)
        masks_probs_raw = item["masks_probs_raw"].to(device)

        anomaly_result = anomaly_score_from_outputs(
            result_logits, cls_logits_raw, masks_probs_raw, T, method
        )
        ood_gts_list.append(item["ood_gts"])
        anomaly_score_list.append(anomaly_result)

        del result_logits, cls_logits_raw, masks_probs_raw
        torch.cuda.empty_cache()

    ood_gts = np.array(ood_gts_list)
    anomaly_scores = np.array(anomaly_score_list)

    ood_out = anomaly_scores[ood_gts == 1]
    ind_out = anomaly_scores[ood_gts == 0]

    val_out = np.concatenate((ind_out, ood_out))
    val_label = np.concatenate((np.zeros(len(ind_out)), np.ones(len(ood_out))))

    prc_auc = average_precision_score(val_label, val_out)
    fpr = fpr_at_95_tpr(val_out, val_label)
    return prc_auc, fpr

In [ ]:
# ---- Configuration for the full table ----
TABLE_METHOD   = "MSP"   # the PDF temperature table uses MSP
TABLE_DATASETS = ["RoadAnomaly21", "RoadObsticle21", "FS_LostFound_full", "fs_static", "RoadAnomaly"]
# rows of the table: fixed temperatures + "best"
TABLE_TEMPS    = ["none", 0.5, 0.75, 1.1, "best"]   # "none" == MSP baseline (T=1.0)

# nice column labels for printing
COL_LABELS = {
    "RoadAnomaly21":     "SMIYC RA-21",
    "RoadObsticle21":    "SMIYC RO-21",
    "FS_LostFound_full": "FS L&F",
    "fs_static":         "FS Static",
    "RoadAnomaly":       "Road Anomaly",
}

# results[temp_label][dataset] = (auprc, fpr, T_used)
results = {}

for ds in TABLE_DATASETS:
    print(f"\n=== Caching {ds} ===")
    cache = cache_dataset(ds)
    print(f"  cached {len(cache)} OoD images")

    # pre-compute the per-dataset best T once (reuses the same cache)
    best_T, best_auprc, best_fpr = None, -1.0, None
    for T in T_GRID:
        a, f = evaluate_cached(cache, float(T), TABLE_METHOD)
        if (a > best_auprc) or (a == best_auprc and f < best_fpr):
            best_T, best_auprc, best_fpr = float(T), a, f

    for temp in TABLE_TEMPS:
        if temp == "none":
            T = 1.0
            a, f = evaluate_cached(cache, T, TABLE_METHOD)
        elif temp == "best":
            T = best_T
            a, f = best_auprc, best_fpr
        else:
            T = float(temp)
            a, f = evaluate_cached(cache, T, TABLE_METHOD)
        results.setdefault(temp, {})[ds] = (a * 100.0, f * 100.0, T)
        print(f"  {ds:18s} temp={str(temp):5s} (T={T:<5}) AuPRC={a*100:.4f}%  FPR95={f*100:.4f}%")

    del cache
    torch.cuda.empty_cache()

In [ ]:
# ---- Pretty-print the table (AuPRC | FPR95 per dataset) ----
row_name = {"none": f"{TABLE_METHOD}", 0.5: f"{TABLE_METHOD}(t=0.5)",
            0.75: f"{TABLE_METHOD}(t=0.75)", 1.1: f"{TABLE_METHOD}(t=1.1)",
            "best": f"{TABLE_METHOD} (best t)"}

header_cells = []
for ds in TABLE_DATASETS:
    header_cells.append(f"{COL_LABELS[ds]:^21s}")
print(f"{'Method':<18s} | " + " | ".join(header_cells))
sub = " " * 18 + " | " + " | ".join([f"{'AuPRC':>9s} {'FPR95':>10s}" for _ in TABLE_DATASETS])
print(sub)
print("-" * len(sub))

for temp in TABLE_TEMPS:
    cells = []
    for ds in TABLE_DATASETS:
        a, f, T = results[temp][ds]
        cells.append(f"{a:9.4f} {f:10.4f}")
    label = row_name.get(temp, str(temp))
    print(f"{label:<18s} | " + " | ".join(cells))

# also show which T was chosen as 'best' per dataset
print("\nBest temperature chosen per dataset:")
for ds in TABLE_DATASETS:
    print(f"  {COL_LABELS[ds]:14s}: T = {results['best'][ds][2]}")

In [ ]:
# ---- Save the table to CSV for the report ----
import csv
os.makedirs('results', exist_ok=True)
csv_path = f'results/Step8_temperature_table_{args.model}_{TABLE_METHOD}.csv'
with open(csv_path, 'w', newline='') as fcsv:
    w = csv.writer(fcsv)
    head = ["Method"]
    for ds in TABLE_DATASETS:
        head += [f"{COL_LABELS[ds]} AuPRC", f"{COL_LABELS[ds]} FPR95"]
    w.writerow(head)
    for temp in TABLE_TEMPS:
        label = row_name.get(temp, str(temp))
        row = [label]
        for ds in TABLE_DATASETS:
            a, f, T = results[temp][ds]
            row += [f"{a:.4f}", f"{f:.4f}"]
        w.writerow(row)
print(f"Saved {csv_path}")